# Ensemble Learning Challenge: The Churn Prediction Cup

## A student model laboratory

In the previous lesson, a single decision tree divided customers using a sequence of rules. Today your team will build several ensembles, tune them, and submit predictions for an **unlabeled challenge set**.

### Central question

> Can your team produce the probability ranking that captures the greatest share of churners in the top 20% of customers?

### Learning goals

By the end of class, you should be able to:

- explain why combining several models can outperform one model;
- distinguish bagging from boosting;
- fit bagging, random forest, AdaBoost, and gradient-boosting classifiers;
- experiment with important ensemble hyperparameters;
- compare training accuracy, test accuracy, cumulative gain, and lift;
- create a valid probability submission for unseen customers;
- recommend a churn model using a business targeting constraint.



In [1]:
import importlib.util
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from sklearn.ensemble import (
    AdaBoostClassifier,
    BaggingClassifier,
    GradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

sns.set_theme(style="whitegrid")

## 1. Human ensemble: can several imperfect opinions help?

Five models estimate the probability that the same customer will churn:

| Model | Probability of churn |
|---|---:|
| A | 0.20 |
| B | 0.45 |
| C | 0.55 |
| D | 0.65 |
| E | 0.80 |

With your group, answer before using Python:

1. Using a 0.50 threshold, what class does each model predict?
2. What does a **hard vote**—majority of predicted classes—decide?
3. What does a **soft vote**—average of probabilities followed by a 0.50 threshold—decide?
4. What could go wrong if all five models make very similar mistakes?

In [ ]:
probabilities = np.array([0.20, 0.45, 0.55, 0.65, 0.80])

# Complete these calculations after your group makes its prediction.
# hard_predictions = ...
# hard_vote = ...
# mean_probability = ...
# soft_vote = ...

### Key idea

An ensemble is most useful when its members are individually helpful **and** make somewhat different errors. Combining five nearly identical models adds less information than combining diverse models.

Today we will study two broad strategies:

| Strategy | Basic idea | Examples |
|---|---|---|
| **Bagging** | Fit models independently to different bootstrap samples, then average or vote | Bagging, random forest |
| **Boosting** | Fit models sequentially so later models focus on earlier errors | AdaBoost, gradient boosting |

<div style="display: flex; gap: 20px; align-items: flex-start;">
  <img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/c/c8/Ensemble_Bagging.svg/1280px-Ensemble_Bagging.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail" alt="First image" style="width: 48%;">
  <img src="https://thumb.wikimedia.org/wikipedia/commons/thumb/b/b5/Ensemble_Boosting.svg/500px-Ensemble_Boosting.svg.png?utm_source=en.wikipedia.org&utm_campaign=parser&utm_content=thumbnail" alt="Second image" style="width: 48%;">
</div>

## 2. Contest launch: training and challenge data

Your team receives two files:

- `ensemble_contest_train.csv`: predictors plus the known `churn` outcome;
- `ensemble_contest_challenge.csv`: predictors for new customers with the outcome hidden.

Upload both files into the same Colab or Jupyter working directory as this notebook. The instructor retains the answer key.

In [ ]:
TRAIN_FILE = "ensemble_contest_train.csv"
CHALLENGE_FILE = "ensemble_contest_challenge.csv"

try:
    churn = pd.read_csv(TRAIN_FILE)
    challenge = pd.read_csv(CHALLENGE_FILE)
except FileNotFoundError as error:
    raise FileNotFoundError(
        "Upload ensemble_contest_train.csv and ensemble_contest_challenge.csv "
        "to the notebook working directory, then rerun this cell."
    ) from error

display(churn.head())
display(challenge.head())

In [ ]:
if churn["churn"].dtype == "object":
    y = (
        churn["churn"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"yes": 1, "true": 1, "1": 1, "no": 0, "false": 0, "0": 0})
    )
else:
    y = churn["churn"].astype(int)

feature_columns = (
    churn
    .drop(columns=["churn", "customer_id"])
    .select_dtypes(include="number")
    .columns
    .tolist()
)

X = churn[feature_columns]
X_challenge = challenge[feature_columns]
challenge_customer_ids = challenge["customer_id"]

print("Predictor shape:", X.shape)
print("Churn rate:", y.mean())
print("Numerical predictors:", X.columns.tolist())

### Contest rules

- **Primary score:** gain at 20% (equivalently, lift at 20%).
- **Tiebreaker:** ROC-AUC across the full challenge set.
- Submit **probabilities**, not zero/one class predictions.
- Every row and every `customer_id` must appear exactly once.
- Each team receives **eight tuning tokens**: one token for each additional model configuration tested after the four starter models.
- Each team may make at most **three official submissions**.
- Do not attempt to recover or inspect the hidden challenge labels.

Awards can recognize:

1. highest gain at 20%;
2. best overall probability ranking;
3. simplest strong model;
4. clearest model explanation.

### Fair-comparison rule

Every model must use:

- the same predictor columns;
- the same training observations;
- the same test observations;
- only training data when fitting;
- the positive-class probability for lift and gain.

Why would changing the test set for each model make the comparison untrustworthy?

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y,
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Test majority-class baseline:", y_test.value_counts(normalize=True).max())

## 3. Bootstrap sampling: the engine behind bagging

A bootstrap sample is created by drawing from the training data **with replacement**. It has the same number of rows as the original sample, but some customers appear more than once and others do not appear at all.

### Predict first

If the training data contain 100 customers and we draw a bootstrap sample of size 100:

- Will all 100 original customers necessarily appear?
- Why might trees trained on different bootstrap samples learn different rules?
- How could averaging those trees stabilize predictions?

In [ ]:
small_training_sample = X_train[["day_mins", "custserv_calls"]].head(10).copy()
small_training_sample["churn"] = y_train.head(10).to_numpy()
small_training_sample

In [ ]:
bootstrap_1 = small_training_sample.sample(n=10, replace=True, random_state=1)
bootstrap_2 = small_training_sample.sample(n=10, replace=True, random_state=2)

display(bootstrap_1)
display(bootstrap_2)

## 4. What counts as the best model?

Accuracy answers: *How often did the model assign the correct class at its current threshold?*

Our business question is different: *How effectively does the model rank churners near the top?*

At a targeting fraction $q$:

$$
\text{Gain}(q)=\frac{\text{churners found in the top }q}{\text{all churners}}
$$

$$
\text{Lift}(q)=\frac{\text{Gain}(q)}{q}
$$

We will rank models primarily by **lift at 20%**, while checking test accuracy and the train–test gap for signs of overfitting.

In [ ]:
def gain_lift_at_fraction(y_true, positive_probability, fraction=0.20):
    results = pd.DataFrame({
        "actual": np.asarray(y_true, dtype=int),
        "probability": positive_probability,
    }).sort_values("probability", ascending=False)

    number_contacted = max(1, int(np.ceil(fraction * len(results))))
    contacted = results.iloc[:number_contacted]
    actual_fraction_contacted = number_contacted / len(results)
    gain = contacted["actual"].sum() / results["actual"].sum()
    lift = gain / actual_fraction_contacted

    return gain, lift


def evaluate_model(name, model, model_type):
    model.fit(X_train, y_train)
    positive_probability = model.predict_proba(X_test)[:, 1]
    gain_20, lift_20 = gain_lift_at_fraction(y_test, positive_probability, 0.20)

    result = {
        "model": name,
        "model_type": model_type,
        "train_accuracy": model.score(X_train, y_train),
        "test_accuracy": model.score(X_test, y_test),
        "gain_at_20": gain_20,
        "lift_at_20": lift_20,
        "train_test_gap": model.score(X_train, y_train) - model.score(X_test, y_test),
    }
    return result, model

## 5. Establish two non-ensemble baselines

Fit:

1. logistic regression;
2. one decision tree with `max_depth=4`.

These models answer whether an ensemble improves on approaches already studied.

We use an explicit `Pipeline` rather than `make_pipeline`. Each tuple gives a step a readable name:

```python
Pipeline([
    ("imputer", SimpleImputer(...)),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(...)),
])
```

This makes the workflow visible and lets you inspect or tune a step with expressions such as `model.named_steps["model"]` or parameter names such as `model__n_estimators`.

In [ ]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])

single_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", DecisionTreeClassifier(max_depth=4, criterion="entropy", random_state=42)),
])

experiment_results = []
fitted_models = {}

for name, model in {
    "logistic regression": logistic_model,
    "single decision tree": single_tree,
}.items():
    result, fitted_model = evaluate_model(name, model, "baseline")
    experiment_results.append(result)
    fitted_models[name] = fitted_model

pd.DataFrame(experiment_results)

### Baseline discussion

- Which baseline has the best test accuracy?
- Which has the best lift at 20%?
- Do the two metrics choose the same model?
- Does either model have a concerning train–test gap?

## 6. Ensemble team laboratory

Every team will build **all four ensemble models**. For each model family:

1. run the starter configuration;
2. change at least one important hyperparameter;
3. run the revised configuration;
4. keep the version with the stronger test lift at 20%;
5. check that the improvement is not accompanied by an unreasonable train–test gap.

This gives every team direct experience with both bagging and boosting. Divide the first four models among group members, then review and revise each model together.

Use the official documentation when deciding which parameter to vary:

- [`BaggingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.BaggingClassifier.html)
- [`RandomForestClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html)
- [`AdaBoostClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html)
- [`GradientBoostingClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html)

### Model 1 — Bagging

Start with shallow decision trees and explore:

- `n_estimators`: 10, 50, 200;
- `max_samples`: 0.5, 0.75, 1.0;
- base-tree `max_depth`: 3, 5, or unrestricted.

Question: Does averaging more trees improve test lift or mainly increase computation?

In [ ]:
bagging_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", BaggingClassifier(
        estimator=DecisionTreeClassifier(max_depth=5, random_state=42),
        n_estimators=50,
        max_samples=0.75,
        random_state=42,
        n_jobs=-1,
    )),
])

# Change parameters and rerun this cell for each experiment.
bagging_result, bagging_model = evaluate_model("bagging", bagging_model, "ensemble")
bagging_result

### Model 2 — Random forest

Explore:

- `n_estimators`: 50, 200, 500;
- `max_depth`: 3, 6, 10, or unrestricted;
- `max_features`: `"sqrt"`, `"log2"`, or a fraction such as 0.5;
- `min_samples_leaf`: 1, 5, or 10.

Question: Why does randomly limiting the features considered at each split create more diverse trees?

In [ ]:
random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        max_features="sqrt",
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1,
    )),
])

random_forest_result, random_forest_model = evaluate_model(
    "random forest", random_forest_model, "ensemble"
)
random_forest_result

### Model 3 — AdaBoost

Explore:

- `n_estimators`: 25, 50, 100, 200;
- `learning_rate`: 0.05, 0.1, 0.5, 1.0;
- shallow base trees with `max_depth` 1, 2, or 3.

Question: What tradeoff do you observe between learning rate and number of estimators?

In [ ]:
adaboost_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
        n_estimators=100,
        learning_rate=0.5,
        random_state=42,
    )),
])

adaboost_result, adaboost_model = evaluate_model(
    "AdaBoost", adaboost_model, "ensemble"
)
adaboost_result

### Model 4 — Gradient boosting

Explore:

- `n_estimators`: 50, 100, 200;
- `learning_rate`: 0.03, 0.05, 0.1, 0.2;
- `max_depth`: 1, 2, or 3;
- `subsample`: 0.6, 0.8, or 1.0.

Question: Can you improve test lift without creating a large train–test accuracy gap?

In [ ]:
gradient_boosting_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=2,
        subsample=1.0,
        random_state=42,
    )),
])

gradient_boosting_result, gradient_boosting_model = evaluate_model(
    "gradient boosting", gradient_boosting_model, "ensemble"
)
gradient_boosting_result

### Team experiment record

Record the starter and revised version of every ensemble. If time permits, test a third configuration for the most promising family.

| Model | Version | Parameters changed | Train accuracy | Test accuracy | Gain at 20% | Lift at 20% |
|---|---|---|---:|---:|---:|---:|
| Bagging | Starter | | | | | |
| Bagging | Revised | | | | | |
| Random forest | Starter | | | | | |
| Random forest | Revised | | | | | |
| AdaBoost | Starter | | | | | |
| AdaBoost | Revised | | | | | |
| Gradient boosting | Starter | | | | | |
| Gradient boosting | Revised | | | | | |

> **Most promising ensemble family:**  
> **Parameter change with the clearest effect:**  
> **Evidence of or against overfitting:**

## 7. Compare your team's four ensembles

After choosing the stronger version from each family, update the four model cells above so they contain your team's selected configurations. The table below keeps a fixed model order; it is a comparison tool, not a competition between teams.

In [ ]:
team_ensemble_models = {
    "bagging": bagging_model,
    "random forest": random_forest_model,
    "AdaBoost": adaboost_model,
    "gradient boosting": gradient_boosting_model,
}

# Keep the two baseline results and evaluate this team's four selected ensembles.
experiment_results = [r for r in experiment_results if r["model_type"] == "baseline"]

for name, model in team_ensemble_models.items():
    result, fitted_model = evaluate_model(name, model, "ensemble")
    experiment_results.append(result)
    fitted_models[name] = fitted_model

model_comparison = pd.DataFrame(experiment_results).reset_index(drop=True)
model_comparison

### Accuracy versus lift

Make a scatterplot with test accuracy on the horizontal axis and lift at 20% on the vertical axis.

- The upper-right area is desirable.
- A model can have slightly lower accuracy but much better lift.
- A large train–test gap is a warning even when lift looks strong.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=model_comparison,
    x="test_accuracy",
    y="lift_at_20",
    hue="model_type",
    s=160,
)

for _, row in model_comparison.iterrows():
    plt.text(row["test_accuracy"] + 0.001, row["lift_at_20"] + 0.01, row["model"], fontsize=9)

plt.title("Which models combine accuracy with lift at 20%?");

## 8. Plot cumulative gain and lift with `scikitplot`

We will compare:

- logistic regression;
- the single decision tree;
- the ensemble with the highest lift at 20%.

The first run may install `scikit-plot` in the notebook environment.

In [ ]:
if importlib.util.find_spec("scikitplot") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-plot"])

# Compatibility for older scikit-plot releases used with newer SciPy versions.
import scipy
if not hasattr(scipy, "interp"):
    scipy.interp = np.interp

import scikitplot as skplt

In [ ]:
ensemble_results = model_comparison.loc[model_comparison["model_type"] == "ensemble"]
best_ensemble_name = ensemble_results.loc[ensemble_results["lift_at_20"].idxmax(), "model"]

comparison_names = [
    "logistic regression",
    "single decision tree",
    best_ensemble_name,
]

print("Best ensemble by lift at 20%:", best_ensemble_name)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, name in zip(axes, comparison_names):
    probabilities = fitted_models[name].predict_proba(X_test)
    skplt.metrics.plot_cumulative_gain(
        y_test,
        probabilities,
        ax=ax,
        title=f"{name}: cumulative gain",
    )
    ax.axvline(0.20, color="black", linestyle=":")

plt.tight_layout();

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, name in zip(axes, comparison_names):
    probabilities = fitted_models[name].predict_proba(X_test)
    skplt.metrics.plot_lift_curve(
        y_test,
        probabilities,
        ax=ax,
        title=f"{name}: lift",
    )
    ax.axvline(0.20, color="black", linestyle=":")

plt.tight_layout();

### Curve-reading questions

1. At 20%, approximately what proportion of churners does each model capture?
2. At 20%, how many times better than random is each model?
3. Does the best model at 20% remain best at 10% or 40%?
4. Why must a model recommendation specify the fraction of customers the company can contact?

## 9. Build your contest submission

Your final submission may use one model or a weighted blend. A blend averages probability estimates from models that may make different errors.

### Strategy meeting

Before producing challenge predictions, decide:

1. Which model has the strongest validation lift at 20%?
2. Which models appear to make meaningfully different predictions?
3. Will you submit one model or a blend?
4. If blending, what weights will you assign and why?

**Do not tune weights using challenge outcomes.** Those labels are hidden.

In [ ]:
# Edit these weights. They must be nonnegative and sum to a positive number.
# A weight of 0 excludes a model. A weight of 1 for one model creates a
# single-model submission.
blend_weights = {
    "bagging": 0.20,
    "random forest": 0.35,
    "AdaBoost": 0.15,
    "gradient boosting": 0.30,
}

weight_total = sum(blend_weights.values())
blend_weights = {name: weight / weight_total for name, weight in blend_weights.items()}
blend_weights

In [ ]:
# Refit each selected model using all labeled contest-training rows, then
# predict probabilities for the unseen challenge customers.
challenge_probability = np.zeros(len(X_challenge))

for name, weight in blend_weights.items():
    model = team_ensemble_models[name]
    model.fit(X, y)
    challenge_probability += weight * model.predict_proba(X_challenge)[:, 1]

TEAM_NAME = "replace-with-team-name"
team_slug = "".join(character if character.isalnum() else "_" for character in TEAM_NAME.lower()).strip("_")

submission = pd.DataFrame({
    "customer_id": challenge_customer_ids,
    "probability_churn": challenge_probability,
})

submission_filename = f"{team_slug}_ensemble_submission.csv"
submission.to_csv(submission_filename, index=False)
submission.head()

### Submission quality control

Run these checks before submitting. A file that fails them cannot be scored.

In [ ]:
assert TEAM_NAME != "replace-with-team-name", "Enter your team name."
assert len(submission) == len(challenge), "Submission row count is incorrect."
assert submission["customer_id"].is_unique, "customer_id values must be unique."
assert set(submission["customer_id"]) == set(challenge["customer_id"]), "customer_id values do not match."
assert submission["probability_churn"].between(0, 1).all(), "Probabilities must be between 0 and 1."
assert submission["probability_churn"].nunique() > 1, "All predictions are identical."

print("Submission passed all checks:", submission_filename)

In [ ]:
# In Google Colab, this downloads the submission to your computer.
try:
    from google.colab import files
    files.download(submission_filename)
except ImportError:
    print("Submit this file from your notebook folder:", submission_filename)

### Where to submit predictions

Official predictions are submitted through the **Ensemble Contest Submission Google Form**, not through the exit ticket and not through Slack.

1. Run the download cell above to save your team's CSV.
2. Open the contest-submission form and provide all required files and answers



- SUBMISSION FORM: [HERE](https://docs.google.com/forms/d/e/1FAIpQLSfBxLSiFCRtUxU-ZIJBmpsQgKWN2pM--SPCpT8wuYRDyiV44Q/viewform?usp=sharing&ouid=106596295392954890759)

## 10. Advanced extension: XGBoost and LightGBM

Teams that complete the four required ensembles may unlock two widely used gradient-boosting libraries:

- **XGBoost** adds regularization and efficient tree boosting.
- **LightGBM** grows trees efficiently and is designed for larger tabular datasets.

These are extensions, not replacements for understanding the four scikit-learn ensembles. Installing them may take a minute.

In [ ]:
for module_name, package_name in [("xgboost", "xgboost"), ("lightgbm", "lightgbm")]:
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [ ]:
xgboost_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1,
    )),
])

lightgbm_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", LGBMClassifier(
        n_estimators=300,
        num_leaves=15,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    )),
])

for name, model in {"XGBoost": xgboost_model, "LightGBM": lightgbm_model}.items():
    result, fitted_model = evaluate_model(name, model, "advanced ensemble")
    display(pd.Series(result))

### Extension challenge

Try changing one parameter in each advanced model. If one improves validation lift, you may use it alone or add it to your blend. Clearly record the model and weights in your submission description.

## Optional extensions

- Add categorical predictors with a `ColumnTransformer` and compare the new lift.
- Compare `RandomForestClassifier` with `ExtraTreesClassifier`.
- Use cross-validation to make model selection less dependent on one split.
- Compare lift at several campaign budgets rather than only 20%.
- Add a cost for each retention offer and a benefit for each successfully retained customer.